In [7]:
! pip -q install langchain_core langchain_openai langgraph

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.7/87.7 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.0/503.0 kB 13.3 MB/s eta 0:00:00


In [1]:
import operator
from typing_extensions import Optional, Annotated, List, Sequence

from langchain_core.messages import BaseMessage
from langgraph.graph import MessagesState
from langgraph.graph.message import add_messages
from pydantic import BaseModel, Field

In [3]:
from datetime import datetime
from typing_extensions import Literal

from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, AIMessage, get_buffer_string
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command


#Base Prompt

In [11]:
clarify_prompt="""                                                                                                               │
│  These are the messages that have been exchanged so far from the user asking for the report:                    │
│  <Messages>                                                                                                     │
│  {messages}                                                                                                     │
│  </Messages>                                                                                                    │
│                                                                                                                 │
│  Today's date is {date}.                                                                                        │
│                                                                                                                 │
│  Assess whether you need to ask a clarifying question, or if the user has already provided enough information   │
│  for you to start research.                                                                                     │
│  IMPORTANT: If you can see in the messages history that you have already asked a clarifying question, you       │
│  almost always do not need to ask another one. Only ask another question if ABSOLUTELY NECESSARY.               │
│                                                                                                                 │
│  If there are acronyms, abbreviations, or unknown terms, ask the user to clarify.                               │
│  If you need to ask a question, follow these guidelines:                                                        │
│  - Be concise while gathering all necessary information                                                         │
│  - Make sure to gather all the information needed to carry out the research task in a concise, well-structured  │
│  manner.                                                                                                        │
│  - Use bullet points or numbered lists if appropriate for clarity. Make sure that this uses markdown            │
│  formatting and will be rendered correctly if the string output is passed to a markdown renderer.               │
│  - Don't ask for unnecessary information, or information that the user has already provided. If you can see     │
│  that the user has already provided the information, do not ask for it again.                                   │
│                                                                                                                 │
│  Respond in valid JSON format with these exact keys:                                                            │
│  "need_clarification": boolean,                                                                                 │
│  "question": "<question to ask the user to clarify the report scope>",                                          │
│  "verification": "<verification message that we will start research>"                                           │
│                                                                                                                 │
│  If you need to ask a clarifying question, return:                                                              │
│  "need_clarification": true,                                                                                    │
│  "question": "<your clarifying question>",                                                                      │
│  "verification": ""                                                                                             │
│                                                                                                                 │
│  If you do not need to ask a clarifying question, return:                                                       │
│  "need_clarification": false,                                                                                   │
│  "question": "",                                                                                                │
│  "verification": "<acknowledgement message that you will now start research based on the provided               │
│  information>"                                                                                                  │
│                                                                                                                 │
│  For the verification message when no clarification is needed:                                                  │
│  - Acknowledge that you have sufficient information to proceed                                                  │
│  - Briefly summarize the key aspects of what you understand from their request                                  │
│  - Confirm that you will now begin the research process                                                         │
│  - Keep the message concise and professional """

#States

In [4]:
class AgentInput(MessagesState):
  'Only Contains messages from user input.'
  pass


class AgentState(MessagesState):
  'Main State For The Multi-Agent System'

  research_brief: Optional[str]
  supervisor_messages: Annotated[Sequence[BaseMessage],add_messages]
  raw_notes: Annotated[list[str],operator.add]=[] #i use this to say that my list will start empty
  notes: Annotated[list[str],operator.add]=[]
  final_report: str

In [5]:
class ClarifyWithUser(BaseModel):
  need_clarification: bool =Field(description='Whether the user needs to be asked a clarifying question.')
  querstion: str=Field(description='The question to be asked to the user.')
  veridication: str=Field(description='The verification in order to start the research.')

class ResearchQuestion(BaseModel):
  research_brief: str=Field(description='The research question.')

# My Model

In [8]:
from langchain_openai import ChatOpenAI

In [9]:
f=open("/content/Nvidia_api_new.txt","r")
nvidia_api_key=f.read()

In [10]:
llm = ChatOpenAI(
    model="meta/llama-3.1-70b-instruct",
    api_key=nvidia_api_key,
    base_url="https://integrate.api.nvidia.com/v1",
)